# RFP LIVE(나라장터 G2B) Dev 100건 전체 빌드 (Kaggle)
(전체 Live-Dev 처리 ->
품질 리포트 -> handoff package 생성)를 수행합니다.

**전제**: `RFP_LIVE_DEV_Smoke_Kaggle.ipynb`로 돌린 5건 스모크가 이미 PASS했습니다(Contract Validation
PASS, Retrieval Scope Violation 0, Generation+Citation Smoke 4/5 - 나머지 1건도 할루시네이션/Hard
Scope 위반이 아니라 포맷 완결성 이슈로 확인). 가이드 12절 "Smoke PASS 후 전체 Live-Dev로 확대"에 따라
**동일한 parser/canonicalizer/block builder/C0 chunker/schema**를 그대로 Live-Dev 100건 중 primary
문서가 선정된 **98건 전체**에 적용합니다(2건은 ZIP 첨부만 있어 3단계에서 이미 `NEEDS_REVIEW`로 보류됨 -
임의로 처리하지 않고 그대로 제외).

**이 노트북은 Retrieval/Generation 인덱스를 다시 만들지 않습니다.** 가이드 15절의 최종 산출물 트리에는
`bm25_index.pkl`/`dense_index.faiss`/`rag_benchmark.jsonl` 같은 인덱스 파일이 포함되어 있지 않습니다 -
Retrieval/Generation 호환성은 이미 3~5건 스모크로 증명했고(가이드 다이어그램: "PASS -> 전체 Live-Dev
변환 -> **팀 Retrieval/Generation 전달**"), 이 노트북의 역할은 **Canonical Documents/Blocks/C0
Chunks + 품질 리포트 + handoff package**까지입니다.

## 입력
`live_collect/prepare_kaggle_live_full_input.py`로 만든 **`LIVE_full_input_v0.1.zip`**(primary 문서
98건 + 짧은 파일명 매핑 `live_dev_full_sample_v0.1.jsonl` + provenance sidecar 2종)을 Kaggle Add
Input으로 연결하세요. Notebook Settings에서 **Internet을 반드시 켜야** 합니다(hwp-hwpx-parser 등
최초 설치 필요). GPU는 이 노트북에선 필수는 아니지만(임베딩/생성 모델을 쓰지 않음) 켜둬도 무방합니다.

## 재사용 원칙
파서(HWP 몽키패치 포함)/정제/청킹 코드는 스모크 노트북과 **완전히 동일**합니다 - 5건에서 검증된 로직을
98건에 그대로 적용하는 것이 이 단계의 핵심이라, 여기서 로직을 바꾸지 않습니다(Holdout
결과를 본 뒤 같은 Holdout에 맞춰 수정하지 않는다"는 원칙으로, 스모크 통과 후 전체 확대
단계에서도 샘플마다 규칙을 바꾸지 않습니다).


In [ ]:
# Kaggle 의존성 설치 (Notebook Settings에서 Internet을 켜고 최초 1회 실행)
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-kor tesseract-ocr-eng > /dev/null
!pip -q install hwp-hwpx-parser pymupdf pytesseract pillow pandas numpy tqdm pyarrow transformers

In [ ]:
# 입력 탐색: Kaggle Add Input으로 연결한 LIVE_full_input_v0.1.zip에서 98건의 hwp/hwpx/pdf와
# live_dev_full_sample_v0.1.jsonl을 찾습니다.
from pathlib import Path
import shutil
import zipfile

KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/live_full_work')
INPUT_ROOT = WORK_ROOT / 'input'
OUTPUT_ROOT = Path('/kaggle/working/live_full_dataset')
IMAGE_DIR = OUTPUT_ROOT / 'images'
for p in (INPUT_ROOT, OUTPUT_ROOT, IMAGE_DIR):
    p.mkdir(parents=True, exist_ok=True)

zip_files = sorted(KAGGLE_INPUT.rglob('*.zip'))
print(f'ZIP 파일 {len(zip_files)}개 발견:', [z.name for z in zip_files])
for zip_path in zip_files:
    extract_dir = INPUT_ROOT / zip_path.stem
    if not extract_dir.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)

SUPPORTED_DOC = {'.hwp', '.hwpx', '.pdf'}
for src in KAGGLE_INPUT.rglob('*'):
    if src.is_file() and (src.suffix.lower() in SUPPORTED_DOC or src.suffix.lower() == '.jsonl'):
        rel = src.relative_to(KAGGLE_INPUT)
        dst = INPUT_ROOT / rel
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

source_files = sorted(p for p in INPUT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED_DOC)
print(f'발견 문서: {len(source_files)}건 (98건이어야 함)')
assert source_files, (
    'HWP/HWPX/PDF 파일을 찾지 못했습니다. LIVE_full_input_v0.1.zip을 Add Input으로 연결했는지 확인하세요.'
)


def find_one(name):
    matches = sorted(INPUT_ROOT.rglob(name)) + sorted(KAGGLE_INPUT.rglob(name))
    return matches[0] if matches else None


full_sample_path = find_one('live_dev_full_sample_v0.1.jsonl')
assert full_sample_path, 'live_dev_full_sample_v0.1.jsonl을 찾지 못했습니다(zip 안에 있어야 합니다).'
print('full sample manifest:', full_sample_path)

In [ ]:
# 공통 유틸: 정규화, 정제, ID 생성 - 스모크 노트북과 동일
import hashlib
import re
import unicodedata as _ud
from datetime import datetime, timezone

SCHEMA_VERSION = '1.0.0'
PARSED_AT = datetime.now(timezone.utc).isoformat()


def nfc(s):
    return _ud.normalize('NFC', str(s)) if s is not None else s


def clean_text(s: str) -> str:
    s = _ud.normalize('NFC', str(s or '')).replace('\x00', '')
    s = s.replace('\r\n', '\n').replace('\r', '\n')
    s = '\n'.join(re.sub(r'[ \t]+', ' ', line).strip() for line in s.split('\n'))
    return re.sub(r'\n{3,}', '\n\n', s).strip()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def stable_id(prefix: str, *parts: str, size: int = 20) -> str:
    raw = '||'.join(nfc(x) for x in parts).encode('utf-8')
    return f'{prefix}_{hashlib.sha256(raw).hexdigest()[:size]}'


HEADING_RE = re.compile(
    r'^(?:제?\s*[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+(?:\.|\s)|제?\s*\d+\s*[장절편.]|\d+(?:\.\d+){0,3}\s*[.)]?\s+'
    r'|[가-힣]\s*[.)]\s+|[【\[].{1,60}[】\]])'
)
LIST_RE = re.compile(r'^(?:[-–—•·※○ㅇ◇◆□■▶▷]|\(?\d+\)|[가-힣]\))\s*')


def classify_text(text: str) -> str:
    t = text.strip()
    if len(t) <= 120 and HEADING_RE.match(t):
        return 'heading'
    if LIST_RE.match(t):
        return 'list_item'
    return 'paragraph'


def section_level(text: str) -> int:
    t = text.strip()
    if re.match(r'^(?:제?\s*[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|제?\s*\d+\s*장)', t):
        return 1
    m = re.match(r'^(\d+(?:\.\d+)*)', t)
    return min(4, m.group(1).count('.') + 2) if m else 2


def attach_context(elements: list) -> list:
    stack = []
    for i, e in enumerate(elements):
        if e['element_type'] == 'heading':
            level = section_level(e['text'])
            stack = stack[:level - 1] + [e['text']]
        e['section_path'] = stack.copy()
        e['header_level'] = section_level(e['text']) if e['element_type'] == 'heading' else None
        e['order_index'] = i
        e['prev_element_id'] = elements[i - 1]['element_id'] if i else None
        e['next_element_id'] = elements[i + 1]['element_id'] if i + 1 < len(elements) else None
    return elements

In [ ]:
# LIVE 메타데이터 로드: G2B bids 필드를 RFP100 documents.jsonl과 동일한 한글 필드명으로 매핑합니다
# (가이드 7절: 값이 없으면 임의로 만들어내지 않습니다).
import json


def read_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


full_sample_rows = read_jsonl(full_sample_path)
filename_to_row = {nfc(row['staged_filename']): row for row in full_sample_rows}
print(f'전체 표본: {len(full_sample_rows)}건')


def parse_budget(row: dict):
    for key in ('asign_bdgt_amt', 'presmpt_prce'):
        raw = row.get(key)
        if raw not in (None, '', 'nan'):
            try:
                return float(str(raw).replace(',', '')), key
            except ValueError:
                continue
    return None, None


def lookup_live_meta(path: Path) -> dict:
    row = filename_to_row.get(nfc(path.name))
    if row is None:
        return {}
    budget, budget_source_field = parse_budget(row)
    return {
        '공고번호': row.get('bid_notice_no'), '공고차수': row.get('bid_notice_ord'),
        '사업명': row.get('bid_notice_nm'),
        '발주기관': row.get('dminstt_nm') or row.get('ntce_instt_nm'),
        '공개일자': row.get('bid_ntce_dt'), '입찰시작일': None, '입찰마감일': row.get('bid_clse_dt'),
        '사업요약': None,
        '사업금액': budget, '_budget_source_field': budget_source_field,
        '_bid_id': row['bid_id'], '_source_url': row.get('source_url'),
        '_pub_prcrmnt_lrg_clsfc_nm': row.get('pub_prcrmnt_lrg_clsfc_nm'),
        '_pub_prcrmnt_mid_clsfc_nm': row.get('pub_prcrmnt_mid_clsfc_nm'),
        '_content_tier': row.get('content_tier'),
    }


matched = sum(bool(lookup_live_meta(p)) for p in source_files)
print(f'LIVE 메타데이터 매칭: {matched}/{len(source_files)}')
assert matched == len(source_files), '전체 98건이 전부 G2B 메타데이터와 매칭되어야 합니다.'

## 11-1. 문서 로딩 및 구조 파싱

스모크 노트북에서 5건으로 검증된 HWP 몽키패치/이미지 필터+OCR/위치보존 파서/HWP·PDF 파서를
**한 글자도 바꾸지 않고** 그대로 98건에 적용합니다.

In [ ]:
# hwp-hwpx-parser 1.0.0 실측 버그 수정 (몽키패치) - 스모크 노트북과 동일
import struct

import hwp_hwpx_parser.hwp5 as _hwp5mod

_CTRL_ID_HYPERLINK = _hwp5mod.CTRL_ID_HYPERLINK


def _patched_extract_hyperlink_texts_from_para(self, para_data):
    hyperlink_texts = []
    i = 0
    while i < len(para_data) - 1:
        code = struct.unpack_from('<H', para_data, i)[0]
        if code == 0x03:
            if i + 6 <= len(para_data):
                ctrl_id = struct.unpack_from('<I', para_data, i + 2)[0]
                if ctrl_id == _CTRL_ID_HYPERLINK:
                    text_start = i + 14
                    text_chars = []
                    j = text_start
                    while j < len(para_data) - 1:
                        c = struct.unpack_from('<H', para_data, j)[0]
                        if c == 0x04:
                            break
                        elif c == 0x03:
                            j += 2
                        elif 0x20 <= c < 0x10000:
                            text_chars.append(chr(c))
                            j += 2
                        else:
                            j += 2
                    if text_chars:
                        hyperlink_texts.append(''.join(text_chars))
                    i = j
                    continue
                else:
                    i += 14
                    continue
            else:
                i += 2
                continue
        elif code == 0x04:
            i += 10
        elif code < 32:
            if code in (11, 12):
                i += 10
            elif 15 <= code <= 23:
                i += 14
            else:
                i += 2
        else:
            i += 2
    return hyperlink_texts


_hwp5mod.HWP5Reader._extract_hyperlink_texts_from_para = _patched_extract_hyperlink_texts_from_para
print('[patch] HWP5Reader._extract_hyperlink_texts_from_para 무한 루프 버그 패치 적용 완료')

In [ ]:
# 이미지 필터링(로고/장식 이미지 제외) + OCR 유틸 - 스모크 노트북과 동일
import io

import numpy as np
import pytesseract
from PIL import Image, ImageFilter

MIN_W, MIN_H = 40, 40
MAX_ASPECT = 15.0
MIN_EDGE_DENSITY = 0.015

_ocr_cache: dict = {}


def edge_density(im: Image.Image) -> float:
    edges = im.convert('L').filter(ImageFilter.FIND_EDGES)
    arr = np.asarray(edges, dtype=np.float32)
    return float((arr > 30).mean())


def image_hash(im: Image.Image) -> str:
    small = im.convert('L').resize((8, 8))
    arr = np.asarray(small, dtype=np.float32)
    bits = arr > arr.mean()
    return ''.join('1' if b else '0' for b in bits.flatten())


def should_ocr(im: Image.Image):
    w, h = im.size
    if w < MIN_W or h < MIN_H:
        return False, 'too_small'
    aspect = max(w, h) / max(1, min(w, h))
    if aspect > MAX_ASPECT:
        return False, 'decorative_aspect'
    if edge_density(im) < MIN_EDGE_DENSITY:
        return False, 'low_edge_density'
    return True, 'ok'


def ocr_image_bytes(data: bytes):
    try:
        im = Image.open(io.BytesIO(data)).convert('RGB')
    except Exception as e:
        return '', (None, None), f'error:decode:{type(e).__name__}'
    h = image_hash(im)
    if h in _ocr_cache:
        text, status = _ocr_cache[h]
        return text, im.size, status
    ok, reason = should_ocr(im)
    if not ok:
        text, status = '', f'skipped:{reason}'
    else:
        try:
            text = clean_text(pytesseract.image_to_string(im, lang='kor+eng', config='--psm 6'))
            status = 'ok'
        except Exception as e:
            text, status = '', f'error:ocr:{type(e).__name__}'
    _ocr_cache[h] = (text, status)
    return text, im.size, status

In [ ]:
# 위치 보존 파서 유틸 - 스모크 노트북과 동일
import re as _re

TABLE_LINE_RE = _re.compile(r'^\|.*\|$')
IMAGE_MARKER_RE = _re.compile(r'\[IMAGE(?::\s*([^\]]*))?\]')


def is_table_unit(unit: str) -> bool:
    lines = [ln.strip() for ln in unit.split('\n') if ln.strip()]
    return bool(lines) and all(TABLE_LINE_RE.match(ln) for ln in lines)


def split_into_raw_elements(raw_text: str, paragraph_separator: str = '\n\n'):
    out = []
    for unit in raw_text.split(paragraph_separator):
        unit = unit.strip('\n')
        if not unit.strip():
            continue
        if is_table_unit(unit):
            out.append(('table', unit, None))
            continue
        last = 0
        found_image = False
        for m in IMAGE_MARKER_RE.finditer(unit):
            found_image = True
            before = unit[last:m.start()]
            if before.strip():
                out.append(('text', before, None))
            out.append(('image', m.group(0), m.group(1)))
            last = m.end()
        if found_image:
            tail = unit[last:]
            if tail.strip():
                out.append(('text', tail, None))
        else:
            out.append(('text', unit, None))
    return out


def markdown_to_rows(md_text: str):
    rows = []
    for line in md_text.split('\n'):
        line = line.strip()
        if not line.startswith('|'):
            continue
        cells_ = [c.strip() for c in line.strip('|').split('|')]
        if all(_re.fullmatch(r':?-{2,}:?', c) for c in cells_):
            continue
        rows.append(cells_)
    return rows


def rows_to_markdown(rows):
    if not rows:
        return ''
    lines = []
    for i, row in enumerate(rows):
        clean_row = [str(c or '').replace('\n', ' ').replace('|', '\\|').strip() for c in row]
        lines.append('| ' + ' | '.join(clean_row) + ' |')
        if i == 0:
            lines.append('| ' + ' | '.join(['---'] * len(clean_row)) + ' |')
    return '\n'.join(lines)

In [ ]:
# HWP/HWPX 파서 - 스모크 노트북과 동일
from hwp_hwpx_parser import ExtractOptions, ImageMarkerStyle, Reader, TableStyle

HWP_OPTIONS = ExtractOptions(
    table_style=TableStyle.MARKDOWN,
    image_marker=ImageMarkerStyle.WITH_NAME,
    paragraph_separator='\n\n',
    line_separator='\n',
)


def parse_hwp(path: Path, canonical_id: str, image_dir: Path):
    elements, tables, cells, images = [], [], [], []
    with Reader(path) as reader:
        result = reader.extract_text_with_notes(HWP_OPTIONS)
        raw_images = reader.get_images()

    image_by_name = {}
    for ii, img in enumerate(raw_images):
        name = nfc(img.filename or f'image_{ii:03d}.{img.format}')
        ext = img.format if img.format != 'unknown' else 'bin'
        saved = image_dir / f'{canonical_id}_{ii:04d}.{ext}'
        saved.write_bytes(img.data)
        ocr_text, (w, h), status = ocr_image_bytes(img.data)
        rec = {
            'image_id': stable_id('img', canonical_id, str(ii)), 'canonical_doc_id': canonical_id,
            'image_index': ii, 'source_name': name, 'saved_path': str(saved.relative_to(OUTPUT_ROOT)),
            'format': img.format, 'width': w, 'height': h, 'ocr_text': ocr_text, 'ocr_status': status,
        }
        images.append(rec)
        image_by_name[name] = rec

    for kind, raw, image_name in split_into_raw_elements(result.text, HWP_OPTIONS.paragraph_separator):
        idx = len(elements)
        eid = stable_id('el', canonical_id, str(idx))
        if kind == 'table':
            rows = markdown_to_rows(raw)
            table_id = stable_id('tbl', canonical_id, str(len(tables)))
            tables.append({
                'table_id': table_id, 'canonical_doc_id': canonical_id, 'table_index': len(tables),
                'row_count': len(rows), 'col_count': len(rows[0]) if rows else 0,
                'markdown': clean_text(raw), 'rows': rows,
            })
            for ri, row in enumerate(rows):
                for ci, value in enumerate(row):
                    cells.append({
                        'cell_id': stable_id('cell', table_id, str(ri), str(ci)), 'table_id': table_id,
                        'canonical_doc_id': canonical_id, 'row_index': ri, 'col_index': ci,
                        'text': clean_text(value),
                    })
            elements.append({
                'element_id': eid, 'canonical_doc_id': canonical_id, 'element_type': 'table',
                'text': clean_text(raw), 'table_id': table_id, 'image_id': None,
                'evidence': {'source_file': nfc(path.name), 'parser': 'hwp-hwpx-parser', 'logical_index': idx},
            })
        elif kind == 'image':
            rec = image_by_name.get(nfc(image_name or ''))
            image_id = rec['image_id'] if rec else None
            text = rec['ocr_text'] if rec else ''
            elements.append({
                'element_id': eid, 'canonical_doc_id': canonical_id, 'element_type': 'image',
                'text': text, 'table_id': None, 'image_id': image_id,
                'evidence': {'source_file': nfc(path.name), 'parser': 'hwp-hwpx-parser', 'logical_index': idx,
                             'image_marker_name': nfc(image_name or '')},
            })
        else:
            text = clean_text(raw)
            if not text:
                continue
            elements.append({
                'element_id': eid, 'canonical_doc_id': canonical_id, 'element_type': classify_text(text),
                'text': text, 'table_id': None, 'image_id': None,
                'evidence': {'source_file': nfc(path.name), 'parser': 'hwp-hwpx-parser', 'logical_index': idx},
            })

    extras = {
        'footnotes': [vars(x) for x in result.footnotes], 'endnotes': [vars(x) for x in result.endnotes],
        'hyperlinks': [list(x) for x in result.hyperlinks], 'memos': [vars(x) for x in result.memos],
    }
    return attach_context(elements), tables, cells, images, extras, result.text

In [ ]:
# PDF 파서 - 스모크 노트북과 동일
import fitz
import tempfile

fitz.TOOLS.mupdf_display_errors(False)


def bbox_overlap_ratio(inner, outer) -> float:
    x0, y0 = max(inner[0], outer[0]), max(inner[1], outer[1])
    x1, y1 = min(inner[2], outer[2]), min(inner[3], outer[3])
    if x1 <= x0 or y1 <= y0:
        return 0.0
    inter = (x1 - x0) * (y1 - y0)
    area = max(1e-6, (inner[2] - inner[0]) * (inner[3] - inner[1]))
    return inter / area


def parse_pdf(path: Path, canonical_id: str, image_dir: Path):
    elements, tables, cells, images = [], [], [], []
    full_text_parts = []
    doc = fitz.open(path)
    try:
        repaired_path = Path(tempfile.gettempdir()) / f'{canonical_id}_repaired.pdf'
        doc.save(str(repaired_path), garbage=4, clean=True, deflate=True)
        doc.close()
        doc = fitz.open(repaired_path)
    except Exception:
        pass
    for page_no, page in enumerate(doc, start=1):
        try:
            table_finder = page.find_tables()
            page_tables = list(table_finder.tables) if table_finder else []
        except ParseTimeout:
            raise
        except Exception:
            page_tables = []

        table_entries = []
        for ti, tbl in enumerate(page_tables):
            rows = tbl.extract()
            if not rows or not any(any(c for c in row) for row in rows):
                continue
            table_id = stable_id('tbl', canonical_id, str(page_no), str(ti))
            md_text = rows_to_markdown(rows)
            tables.append({
                'table_id': table_id, 'canonical_doc_id': canonical_id, 'table_index': len(tables),
                'row_count': len(rows), 'col_count': len(rows[0]) if rows else 0, 'markdown': md_text, 'rows': rows,
            })
            for ri, row in enumerate(rows):
                for ci, value in enumerate(row):
                    cells.append({
                        'cell_id': stable_id('cell', table_id, str(ri), str(ci)), 'table_id': table_id,
                        'canonical_doc_id': canonical_id, 'row_index': ri, 'col_index': ci,
                        'text': clean_text(value or ''),
                    })
            table_entries.append({'bbox': tuple(tbl.bbox), 'table_id': table_id, 'markdown': md_text})
            full_text_parts.append(md_text)

        page_blocks = sorted(page.get_text('blocks'), key=lambda b: (round(b[1], 1), round(b[0], 1)))
        for bi, b in enumerate(page_blocks):
            text = clean_text(b[4])
            if not text:
                continue
            block_bbox = tuple(b[:4])
            if any(bbox_overlap_ratio(block_bbox, t['bbox']) > 0.5 for t in table_entries):
                continue
            idx = len(elements)
            elements.append({
                'element_id': stable_id('el', canonical_id, str(idx)), 'canonical_doc_id': canonical_id,
                'element_type': classify_text(text), 'text': text, 'table_id': None, 'image_id': None,
                'evidence': {'source_file': nfc(path.name), 'parser': 'pymupdf', 'page': page_no,
                             'bbox': [round(float(x), 2) for x in block_bbox], 'page_block_index': bi},
            })
            full_text_parts.append(text)

        for te in table_entries:
            idx = len(elements)
            elements.append({
                'element_id': stable_id('el', canonical_id, str(idx)), 'canonical_doc_id': canonical_id,
                'element_type': 'table', 'text': te['markdown'], 'table_id': te['table_id'], 'image_id': None,
                'evidence': {'source_file': nfc(path.name), 'parser': 'pymupdf.find_tables', 'page': page_no,
                             'bbox': [round(float(x), 2) for x in te['bbox']]},
            })

        for ii, info in enumerate(page.get_images(full=True)):
            xref = info[0]
            img_data = doc.extract_image(xref)
            payload, ext = img_data['image'], img_data.get('ext', 'bin')
            global_i = len(images)
            saved = image_dir / f'{canonical_id}_{global_i:04d}.{ext}'
            saved.write_bytes(payload)
            ocr_text, (w, h), status = ocr_image_bytes(payload)
            image_id = stable_id('img', canonical_id, str(global_i))
            images.append({
                'image_id': image_id, 'canonical_doc_id': canonical_id, 'image_index': global_i,
                'source_name': f'page_{page_no}_image_{ii}', 'saved_path': str(saved.relative_to(OUTPUT_ROOT)),
                'format': ext, 'width': w, 'height': h, 'ocr_text': ocr_text, 'ocr_status': status,
                'page': page_no, 'xref': xref,
            })
            idx = len(elements)
            elements.append({
                'element_id': stable_id('el', canonical_id, str(idx)), 'canonical_doc_id': canonical_id,
                'element_type': 'image', 'text': ocr_text, 'table_id': None, 'image_id': image_id,
                'evidence': {'source_file': nfc(path.name), 'parser': 'pymupdf+pytesseract', 'page': page_no,
                             'xref': xref},
            })
            if ocr_text:
                full_text_parts.append(ocr_text)
    doc.close()

    def sort_key(e):
        ev = e['evidence']
        page = ev.get('page', 0)
        y = ev.get('bbox', [0, 10 ** 9])[1]
        return (page, y, e['element_id'])

    elements.sort(key=sort_key)
    return attach_context(elements), tables, cells, images, {}, '\n\n'.join(full_text_parts)

In [ ]:
# 문서 단위 파싱 실행 - 스모크 노트북과 동일 로직, 전체 98건 대상
# 실패는 SUCCESS로 기록하지 않고 errors.jsonl(=이 노트북에서는 errors 리스트)에 남긴다.
import signal
import traceback
from collections import Counter, defaultdict

from tqdm.auto import tqdm

PARSE_TIMEOUT_SECONDS = 60


class ParseTimeout(Exception):
    pass


def _timeout_handler(signum, frame):
    raise ParseTimeout(f'파싱이 {PARSE_TIMEOUT_SECONDS}초를 넘겨 중단했습니다')


def parse_with_timeout(fn, *args):
    has_alarm = hasattr(signal, 'SIGALRM')
    if not has_alarm:
        return fn(*args)
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(PARSE_TIMEOUT_SECONDS)
    try:
        return fn(*args)
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


file_records = []
for path in source_files:
    payload = path.read_bytes()
    digest = sha256_bytes(payload)
    file_records.append({'path': path, 'file_sha256': digest, 'canonical_doc_id': f'doc_{digest[:20]}',
                          'file_size': len(payload)})
hash_counts = Counter(x['file_sha256'] for x in file_records)
print('물리 원본 중복 그룹:', sum(v > 1 for v in hash_counts.values()))

documents, all_elements, all_tables, all_cells, all_images, errors = [], [], [], [], [], []
parsed_cache = {}
for f in tqdm(file_records, desc='LIVE 전체(98건) 파싱'):
    path, cid = f['path'], f['canonical_doc_id']
    try:
        if cid not in parsed_cache:
            parse_fn = parse_pdf if path.suffix.lower() == '.pdf' else parse_hwp
            result = parse_with_timeout(parse_fn, path, cid, IMAGE_DIR)
            parsed_cache[cid] = result
            elements, tables, cells, images, extras, raw_text = result
            all_elements.extend(elements)
            all_tables.extend(tables)
            all_cells.extend(cells)
            all_images.extend(images)
        else:
            elements, tables, cells, images, extras, raw_text = parsed_cache[cid]

        meta = lookup_live_meta(path)
        budget = meta.pop('사업금액', None)
        documents.append({
            'source_instance_id': stable_id('src', str(path.relative_to(INPUT_ROOT))),
            'canonical_doc_id': cid, 'source_path': nfc(str(path.relative_to(INPUT_ROOT))),
            'source_filename': nfc(path.name), 'extension': path.suffix.lower(),
            'file_sha256': f['file_sha256'], 'file_size': f['file_size'],
            'is_exact_duplicate': hash_counts[f['file_sha256']] > 1,
            'exact_duplicate_count': hash_counts[f['file_sha256']],
            'character_count': len(raw_text), 'element_count': len(elements),
            'table_count': len(tables), 'image_count': len(images), 'extras': extras,
            'csv_metadata_matched': bool(meta.get('_bid_id')),
            '공고번호': meta.get('공고번호'), '공고차수': meta.get('공고차수'), '사업명': meta.get('사업명'),
            '사업금액': budget, '발주기관': meta.get('발주기관'), '공개일자': meta.get('공개일자'),
            '입찰시작일': meta.get('입찰시작일'), '입찰마감일': meta.get('입찰마감일'), '사업요약': meta.get('사업요약'),
            'schema_version': SCHEMA_VERSION, 'parsed_at': PARSED_AT,
            'split': 'LIVE_DEV', 'bid_id': meta.get('_bid_id'), 'metadata_source': 'G2B',
            'budget_status': 'KNOWN' if budget is not None else 'UNKNOWN',
            'budget_source_field': meta.get('_budget_source_field'),
            'parse_status': 'SUCCESS', 'parse_warnings': [],
            'data_contract_version': 'FROZEN_v0.1', 'source_text_version': 'canonical_text_v0.1',
        })
    except Exception as e:
        errors.append({
            'source_path': nfc(str(path.relative_to(INPUT_ROOT))), 'canonical_doc_id': cid,
            'error_type': type(e).__name__, 'message': str(e), 'traceback': traceback.format_exc(limit=5),
        })

canonical_ids = set(d['canonical_doc_id'] for d in documents)
print(f'성공 {len(documents)} / 입력 {len(source_files)}, 오류 {len(errors)}, canonical 문서 {len(canonical_ids)}')
if errors:
    print('오류 상세:', errors)
print(f'\n파싱 성공률: {len(documents)}/{len(source_files)} = {len(documents) / max(1, len(source_files)):.1%}')

## 11-2. 정제(Cleaning) + Blocks 생성

스모크 노트북과 동일한 정제 로직으로 노이즈를 제거하고 문서별로 정렬해 blocks(=elements)를
확정합니다.

In [ ]:
# 제어 아티팩트/머리말·꼬리말/페이지번호 정제 - 스모크 노트북과 동일
MOJIBAKE_RE = re.compile(r'�+|(?:.ȃ)+')
PAGE_NO_RE = re.compile(r'^\s*(?:[-–—]?\s*\d{1,4}\s*[-–—]?|page\s*\d{1,4})\s*$', re.I)
DOT_LEADER_RE = re.compile(r'[.·ㆍ…]{4,}\s*\d{1,4}\s*$')
ALLOWED_CONTROLS = {'\n', '\t'}


def clean_with_audit(text: str):
    raw = _ud.normalize('NFC', text)
    removed, kept = [], []
    for pos, ch in enumerate(raw):
        category = _ud.category(ch)
        if category in {'Cc', 'Cf', 'Cs', 'Co', 'Cn'} and ch not in ALLOWED_CONTROLS:
            removed.append({'position': pos, 'text': repr(ch), 'reason': f'unicode_{category}'})
        else:
            kept.append(ch)
    value = ''.join(kept)

    def drop_mojibake(m):
        removed.append({'position': m.start(), 'text': m.group(0), 'reason': 'decode_artifact'})
        return ' '

    value = MOJIBAKE_RE.sub(drop_mojibake, value)
    value = '\n'.join(DOT_LEADER_RE.sub('', line) for line in value.splitlines())
    value = clean_text(value)
    ratio = round(max(0.0, (len(raw) - len(value)) / max(1, len(raw))), 6)
    return value, removed, ratio


artifact_logs = []
for e in all_elements:
    e['text_raw'] = e['text']
    e['text'], removed, ratio = clean_with_audit(e['text_raw'])
    e['artifact_ratio'] = ratio
    e['needs_review'] = ratio > 0.05 or len(removed) > 10
    if removed:
        artifact_logs.append({'element_id': e['element_id'], 'canonical_doc_id': e['canonical_doc_id'],
                               'removed_count': len(removed), 'artifact_ratio': ratio,
                               'removed': removed[:100]})
    if not e['text']:
        e['noise_label'] = 'empty_after_cleaning'
    elif PAGE_NO_RE.match(e['text']):
        e['noise_label'] = 'page_number'
    else:
        bbox = e['evidence'].get('bbox')
        if bbox and bbox[1] < 45:
            e['noise_label'] = 'pdf_header_candidate'
        elif bbox and bbox[1] > 790:
            e['noise_label'] = 'pdf_footer_candidate'
        else:
            e['noise_label'] = None

short_counter = defaultdict(Counter)
for e in all_elements:
    if 3 <= len(e['text']) <= 100:
        short_counter[e['canonical_doc_id']][e['text']] += 1
NOISE_EXCLUDE = {'page_number', 'empty_after_cleaning', 'pdf_header_candidate', 'pdf_footer_candidate'}
for e in all_elements:
    if e['noise_label'] is None and e['element_type'] == 'paragraph' \
            and short_counter[e['canonical_doc_id']][e['text']] >= 3:
        e['noise_label'] = 'repeated_short_text'
    e['include_in_analysis'] = e['noise_label'] not in (NOISE_EXCLUDE | {'repeated_short_text'})

print('정제 요소(=blocks):', len(all_elements), '| 아티팩트 감지:', len(artifact_logs),
      '| 검토 필요:', sum(e['needs_review'] for e in all_elements),
      '| 분석 제외:', sum(not e['include_in_analysis'] for e in all_elements))

In [ ]:
elements_by_doc = defaultdict(list)
for e in all_elements:
    if e['include_in_analysis']:
        elements_by_doc[e['canonical_doc_id']].append(e)
for cid in elements_by_doc:
    elements_by_doc[cid].sort(key=lambda x: x['order_index'])

doc_meta_by_id = {d['canonical_doc_id']: d for d in documents}
for t in all_tables:
    t['content_type'] = 'table'
    t['serialization'] = 'markdown'
for im in all_images:
    im['content_type'] = 'image_ocr'
    im['caption_candidate'] = im['ocr_text'][:300] if im['ocr_text'] else None

print('블록(elements) 총:', len(all_elements), '| 표:', len(all_tables), '| 이미지:', len(all_images),
      '| OCR 텍스트 있는 이미지:', sum(bool(x['ocr_text']) for x in all_images))

## 11-3. C0 Chunks 생성

스모크 노트북에서 재사용을 확정한 RFP100 baseline 청커(섹션 경계 우선 + 300~800 토큰, BGE-m3
토크나이저)를 그대로 씁니다.

In [ ]:
from transformers import AutoTokenizer

DENSE_MODEL = 'BAAI/bge-m3'
dense_tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL)
MIN_TOKENS, TARGET_TOKENS, MAX_TOKENS, OVERLAP_TOKENS = 300, 600, 800, 60


def token_count(text: str) -> int:
    return len(dense_tokenizer.encode(text, add_special_tokens=False))


def split_long_text(text: str, max_tokens: int = MAX_TOKENS):
    ids = dense_tokenizer.encode(text, add_special_tokens=False)
    parts, start = [], 0
    while start < len(ids):
        end = min(len(ids), start + max_tokens)
        parts.append(dense_tokenizer.decode(ids[start:end], skip_special_tokens=True).strip())
        if end == len(ids):
            break
        start = max(start + 1, end - OVERLAP_TOKENS)
    return [p for p in parts if p]


chunks = []
chunk_seq = defaultdict(int)


def doc_title(cid):
    meta = doc_meta_by_id.get(cid, {})
    return meta.get('사업명') or meta.get('source_filename')


def emit_chunk(cid, buf, kind='text', forced_text=None):
    text = (forced_text if forced_text is not None else '\n\n'.join(x['text'] for x in buf if x['text'])).strip()
    if not text:
        return
    for part in split_long_text(text):
        idx = chunk_seq[cid]
        chunk_seq[cid] += 1
        meta = doc_meta_by_id.get(cid, {})
        chunks.append({
            'chunk_id': stable_id('chk', cid, str(idx)), 'text': part,
            'metadata': {
                'canonical_doc_id': cid, 'chunk_index': idx, 'chunk_type': kind,
                'document_title': doc_title(cid), '발주기관': meta.get('발주기관'),
                '공개일자': meta.get('공개일자'), '사업금액': meta.get('사업금액'),
                'section_path': buf[-1].get('section_path', []) if buf else [],
                'header_level': buf[-1].get('header_level') if buf else None,
                'element_ids': [x['element_id'] for x in buf],
                'order_index_start': min((x['order_index'] for x in buf), default=None),
                'order_index_end': max((x['order_index'] for x in buf), default=None),
                'source_filenames': sorted(set(x['evidence']['source_file'] for x in buf)),
                'needs_review': any(x['needs_review'] for x in buf),
            },
            'token_count': token_count(part),
        })


for cid, doc_elements in tqdm(elements_by_doc.items(), desc='청킹'):
    lookup = {x['element_id']: x for x in doc_elements}
    buf, size = [], 0
    for e in doc_elements:
        special = e['element_type'] in {'table', 'image'}
        et = token_count(e['text'])
        if e['element_type'] == 'heading' and size >= MIN_TOKENS:
            emit_chunk(cid, buf)
            buf, size = [], 0
        if special:
            if buf:
                emit_chunk(cid, buf)
                buf, size = [], 0
            context = [lookup[x] for x in (e.get('prev_element_id'),) if x in lookup] + [e] \
                + [lookup[x] for x in (e.get('next_element_id'),) if x in lookup]
            emit_chunk(cid, context, e['element_type'])
            continue
        if buf and size + et > MAX_TOKENS:
            emit_chunk(cid, buf)
            buf, size = [], 0
        buf.append(e)
        size += et
        if size >= TARGET_TOKENS:
            emit_chunk(cid, buf)
            buf, size = [], 0
    if buf:
        emit_chunk(cid, buf)

for c in chunks:
    m = c['metadata']
    c['retrieval_text'] = ' '.join(filter(None, [
        m.get('document_title'), m.get('발주기관'), ' '.join(m.get('section_path', [])), c['text'],
    ]))

token_counts = [c['token_count'] for c in chunks]
print('청크(C0):', len(chunks), '| 토큰 중앙값:', int(np.median(token_counts)) if chunks else 0,
      '| 800 초과:', sum(t > 800 for t in token_counts))

## 11-4. Contract Validation (98건 전체 대상)

스모크에서 5건으로 검증했던 동일한 체크리스트를 이제 98건 전체에 적용합니다.

In [ ]:
validation_results = []


def check(name, condition, detail=''):
    validation_results.append({'check': name, 'pass': bool(condition), 'detail': detail})


doc_ids_set = set(d['canonical_doc_id'] for d in documents)

check('documents.canonical_doc_id unique', len(doc_ids_set) == len(documents))
check('documents.parse_status 전부 SUCCESS', all(d['parse_status'] == 'SUCCESS' for d in documents))
empty_text_docs = [d['canonical_doc_id'] for d in documents if d['character_count'] == 0]
check('documents: parse_status=SUCCESS인데 character_count=0인 레코드 = 0', len(empty_text_docs) == 0,
      str(empty_text_docs))
check('documents.file_sha256(content hash) 전부 존재', all(bool(d['file_sha256']) for d in documents))
missing_title = [d['canonical_doc_id'] for d in documents if not d.get('사업명')]
check('documents.사업명(title) 존재', len(missing_title) == 0, str(missing_title))
missing_agency = [d['canonical_doc_id'] for d in documents if not d.get('발주기관')]
check('documents.발주기관(agency) 존재', len(missing_agency) == 0, str(missing_agency))
check('documents.bid_id(원본 provenance 추적 가능) 전부 존재', all(bool(d.get('bid_id')) for d in documents))

element_ids_set = set(e['element_id'] for e in all_elements)
check('blocks.element_id unique', len(element_ids_set) == len(all_elements))
orphan_elements = [e['element_id'] for e in all_elements if e['canonical_doc_id'] not in doc_ids_set]
check('blocks.canonical_doc_id FK(documents에 존재)', len(orphan_elements) == 0, str(orphan_elements[:5]))
bad_order = [e['element_id'] for e in all_elements if not isinstance(e.get('order_index'), int)]
check('blocks.order_index 정수 (char_start<char_end 대체 - 구조적 위치 보존)', len(bad_order) == 0,
      str(bad_order[:5]))

chunk_ids_set = set(c['chunk_id'] for c in chunks)
check('chunks.chunk_id unique', len(chunk_ids_set) == len(chunks))
orphan_chunks = [c['chunk_id'] for c in chunks if c['metadata']['canonical_doc_id'] not in doc_ids_set]
check('chunks.canonical_doc_id FK(documents에 존재), orphan = 0', len(orphan_chunks) == 0, str(orphan_chunks[:5]))
empty_chunks = [c['chunk_id'] for c in chunks if not c['text'].strip()]
check('chunks.text empty = 0', len(empty_chunks) == 0, str(empty_chunks[:5]))
bad_element_fk = [c['chunk_id'] for c in chunks
                   if any(eid not in element_ids_set for eid in c['metadata']['element_ids'])]
check('chunks.metadata.element_ids FK(blocks에 존재)', len(bad_element_fk) == 0, str(bad_element_fk[:5]))

element_by_id = {e['element_id']: e for e in all_elements}
cross_doc_chunks = []
for c in chunks:
    cid = c['metadata']['canonical_doc_id']
    if any(element_by_id[eid]['canonical_doc_id'] != cid for eid in c['metadata']['element_ids']
           if eid in element_by_id):
        cross_doc_chunks.append(c['chunk_id'])
check('chunks가 참조하는 blocks가 전부 같은 문서 소속 (cross-doc 오염 = 0)', len(cross_doc_chunks) == 0,
      str(cross_doc_chunks[:5]))

all_pass = all(r['pass'] for r in validation_results)
print(f"{'CHECK':<62}{'RESULT'}")
for r in validation_results:
    status = 'PASS' if r['pass'] else f"FAIL ({r['detail']})"
    print(f"{r['check']:<62}{status}")
print(f"\n전체 Contract Validation (98건): {'PASS' if all_pass else 'FAIL'}")
if not all_pass:
    print('경고: 실패 항목이 있습니다. 원인을 확인한 뒤 handoff 여부를 판단하세요(assert로 강제 중단하지'
          ' 않음 - 98건 규모에서는 일부 FK 예외가 있을 수 있어 리포트에 남기고 팀과 상의하는 쪽을 택함).')

## 11-5 / 12. 품질 리포트 (Quality Report)

`RFP_100_RAG_Pipeline_Kaggle.ipynb`의 `quality_report.json` 방식과 동일하게, 파싱 성공률/요소·표·이미지
개수/청크 토큰 분포/검토 필요 비율을 집계합니다.

In [ ]:
quality_report = {
    'schema_version': SCHEMA_VERSION,
    'input_document_count': len(source_files), 'parsed_source_count': len(documents),
    'canonical_document_count': len(canonical_ids),
    'exact_duplicate_groups': sum(v > 1 for v in hash_counts.values()),
    'csv_metadata_matched_count': sum(d['csv_metadata_matched'] for d in documents),
    'budget_known_count': sum(d['budget_status'] == 'KNOWN' for d in documents),
    'element_count': len(all_elements), 'table_count': len(all_tables), 'table_cell_count': len(all_cells),
    'image_count': len(all_images), 'ocr_nonempty_count': sum(bool(x['ocr_text']) for x in all_images),
    'artifact_log_count': len(artifact_logs),
    'review_element_count': sum(e['needs_review'] for e in all_elements),
    'chunk_count': len(chunks),
    'chunk_token_min': min((x['token_count'] for x in chunks), default=0),
    'chunk_token_median': float(np.median(token_counts)) if token_counts else 0,
    'chunk_token_max': max((x['token_count'] for x in chunks), default=0),
    'error_count': len(errors), 'errors': errors,
    'element_type_counts': dict(Counter(x['element_type'] for x in all_elements)),
    'contract_validation_all_pass': all_pass,
    'checks': {
        'all_elements_have_order': all(isinstance(x.get('order_index'), int) for x in all_elements),
        'all_elements_have_evidence': all(bool(x.get('evidence')) for x in all_elements),
        'all_chunks_have_metadata': all(bool(x.get('metadata')) for x in chunks),
        'chunk_token_max_800': all(x['token_count'] <= 800 for x in chunks),
    },
}
print(json.dumps({k: v for k, v in quality_report.items() if k != 'errors'}, ensure_ascii=False, indent=2))

import pandas as pd

pd.set_option('display.max_colwidth', 50)
doc_preview_cols = ['canonical_doc_id', 'bid_id', '사업명', '발주기관', 'extension', 'character_count',
                     'element_count', 'table_count', 'image_count', 'parse_status']
display(pd.DataFrame(documents)[doc_preview_cols])

## 13. Handoff Package 생성

가이드 15절 파일명/폴더 구조 그대로 저장합니다(이번엔 스모크가 아니라 98건 전체이므로 접미사 없이
`LIVE_DATA_HANDOFF_v0.1`로 저장).

In [ ]:
LIVE_HANDOFF_ROOT = Path('/kaggle/working/LIVE_DATA_HANDOFF_v0.1')
for sub in ('data', 'metadata', 'reports', 'contracts'):
    (LIVE_HANDOFF_ROOT / sub).mkdir(parents=True, exist_ok=True)


def write_jsonl(path, rows):
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')


write_jsonl(LIVE_HANDOFF_ROOT / 'data' / 'LIVE_documents_DEV_v0.1.jsonl', documents)
write_jsonl(LIVE_HANDOFF_ROOT / 'data' / 'LIVE_blocks_DEV_v0.1.jsonl', all_elements)
write_jsonl(LIVE_HANDOFF_ROOT / 'data' / 'LIVE_chunks_C0_DEV_v0.1.jsonl', chunks)

catalog = pd.DataFrame(documents)[
    ['canonical_doc_id', 'bid_id', '공고번호', '사업명', '발주기관', 'source_filename', 'extension', 'parse_status']
].rename(columns={'canonical_doc_id': 'document_id', '사업명': 'title', '발주기관': 'agency'})
catalog.to_csv(LIVE_HANDOFF_ROOT / 'metadata' / 'LIVE_document_catalog_DEV_v0.1.csv', index=False,
               encoding='utf-8-sig')

full_row_by_bid = {r['bid_id']: r for r in full_sample_rows}
live_provenance = []
for d in documents:
    sr = full_row_by_bid.get(d['bid_id'], {})
    live_provenance.append({
        'canonical_doc_id': d['canonical_doc_id'], 'bid_id': d['bid_id'],
        'bid_notice_no': d['공고번호'], 'bid_notice_ord': d['공고차수'], 'notice_date': d['공개일자'],
        'source_url': sr.get('source_url'),
        'pub_prcrmnt_lrg_clsfc_nm': sr.get('pub_prcrmnt_lrg_clsfc_nm'),
        'pub_prcrmnt_mid_clsfc_nm': sr.get('pub_prcrmnt_mid_clsfc_nm'),
        'attachments': [{'filename': d['source_filename'], 'sha256': d['file_sha256'],
                          'processing_status': 'SEARCHABLE'}],
    })
write_jsonl(LIVE_HANDOFF_ROOT / 'metadata' / 'LIVE_provenance_v0.1.jsonl', live_provenance)

field_mapping_md = '''# LIVE_FIELD_MAPPING_v0.1

`RFP_LIVE_DEV_Smoke_Kaggle.ipynb`(3~5건 스모크)에서 확정한 매핑과 동일합니다(이 노트북은 98건 전체에
같은 파이프라인을 적용했을 뿐, 스키마를 바꾸지 않았습니다).

| 가이드 예시 필드 | 실제(RFP100/LIVE 공통) 필드 | 비고 |
|---|---|---|
| `document_id` | `canonical_doc_id` (`doc_<sha256[:20]>`) | content-hash 기반 |
| `content_hash` / `source_sha256` | `file_sha256` | RFP100은 이 둘을 분리하지 않음 |
| `bid_id` | `bid_id` (LIVE 전용 추가 필드) | G2B 공고 단위 식별 |
| Structural Block(`block_id`) | `element_id` (`LIVE_blocks_DEV_v0.1.jsonl`) | 표=Markdown, 이미지=OCR 텍스트로 이미 구조화 |
| `char_start`/`char_end` | `order_index`(block) / `order_index_start`,`order_index_end`+`element_ids`(chunk) | 요소 순서로 위치/근거 보존 |
| C0 `fixed-char-1200-o200-v0.1` | 실제 baseline 청커: 섹션 경계 우선 + 300~800 토큰(BGE-m3) | RFP100 검증 청커 재사용 |
'''
(LIVE_HANDOFF_ROOT / 'contracts' / 'LIVE_FIELD_MAPPING_v0.1.md').write_text(field_mapping_md, encoding='utf-8')

validation_md = ['# LIVE_COMPAT_VALIDATION_v0.1', '', f'전체 표본: {len(documents)}건 (Live-Dev 100건 중'
                  f' primary 선정 98건, 3~5건 스모크는 별도 PASS 완료)', '', '| Check | Result |', '|---|---|']
for r in validation_results:
    validation_md.append(f"| {r['check']} | {'PASS' if r['pass'] else 'FAIL: ' + r['detail']} |")
(LIVE_HANDOFF_ROOT / 'reports' / 'LIVE_COMPAT_VALIDATION_v0.1.md').write_text('\n'.join(validation_md),
                                                                               encoding='utf-8')
(LIVE_HANDOFF_ROOT / 'reports' / 'LIVE_PARSE_QUALITY_REPORT_v0.1.md').write_text(
    '# LIVE_PARSE_QUALITY_REPORT_v0.1\n\n```json\n'
    + json.dumps({k: v for k, v in quality_report.items() if k != 'errors'}, ensure_ascii=False, indent=2)
    + '\n```\n\n## 파싱 오류 상세\n\n```json\n' + json.dumps(errors, ensure_ascii=False, indent=2) + '\n```\n',
    encoding='utf-8')

(LIVE_HANDOFF_ROOT / 'README_FIRST.md').write_text(f'''# LIVE Data Handoff v0.1

가이드(`02_G2B_LIVE_TO_EXISTING_RAG_COMPAT_GUIDE_v0.1.md`) 19절 P0(1~10단계, 3~5건 스모크)가 PASS한
뒤 "다음"(11~13단계)까지 완료한 산출물입니다.

- Live-Dev 100건 중 primary 문서 선정 98건 처리(2건은 ZIP 첨부만 있어 `NEEDS_REVIEW`로 제외 -
  `bid_id: R26BK01579894_000, R26BK01604471_001`)
- 파싱 성공: {len(documents)}/{len(source_files)}건 (오류 {len(errors)}건)
- Contract Validation: {'PASS' if all_pass else 'FAIL - reports/LIVE_COMPAT_VALIDATION_v0.1.md 확인 필요'}
- Retrieval/Generation 호환성은 이 산출물이 아니라 `RFP_LIVE_DEV_Smoke_Kaggle.ipynb`의 3~5건 스모크
  결과로 이미 증명되었습니다(Scope Violation 0, Generation+Citation 4/5 - 나머지 1건도 Hard Scope
  위반이 아닌 포맷 이슈). 이 handoff에는 인덱스 파일을 포함하지 않았습니다 - 가이드 15절 최종 산출물
  트리에 인덱스 파일이 없고, Retrieval/Generation은 이 chunks를 받는 쪽에서 구축하는 것이 원칙입니다.
- Final Holdout 후보 60건은 이 산출물에 전혀 포함되어 있지 않습니다(별도 트랙, 아직 미착수).

## 다음 (14단계)
이 zip을 팀원에게 전달하세요.
''', encoding='utf-8')

manifest = {
    'schema_version': SCHEMA_VERSION, 'stage': 'FULL_LIVE_DEV_STEP_11_TO_13', 'sample_size': len(documents),
    'document_ids': sorted(doc_ids_set), 'contract_validation_all_pass': all_pass,
    'parse_error_count': len(errors), 'parsed_at': PARSED_AT,
}
(LIVE_HANDOFF_ROOT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2),
                                                  encoding='utf-8')

archive = shutil.make_archive('/kaggle/working/LIVE_DATA_HANDOFF_v0.1', 'zip', LIVE_HANDOFF_ROOT)
print('완료:', archive)
print(json.dumps(manifest, ensure_ascii=False, indent=2))